# Renderização Fotorealista com Blender Cycles no Google Colab GPU T4

Este notebook configura o Blender com a engine **Cycles** (renderização fotorealista) otimizada para GPU T4 do Google Colab.

## 🎨 Recursos:
- **Blender 4.x** (última versão estável)
- **Cycles Engine** com ray tracing GPU-accelerated
- **OptiX** para renderização NVIDIA RTX
- Suporte para **animações** e **imagens estáticas**
- **Denoising AI** para reduzir ruído

**⚠️ Importante:**
- Ative o runtime GPU T4: Runtime → Change runtime type → GPU
- O Colab tem limite de tempo (12h) e espaço (~100GB)

## 1. Verificar GPU e Sistema

In [ ]:
# Verificar GPU disponível
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

print("\n" + "="*50)
print("RECURSOS DO SISTEMA")
print("="*50)

# Espaço em disco
!df -h | grep -E 'Filesystem|/content'

# Memória RAM
!free -h

/bin/bash: line 1: nvidia-smi: command not found

RECURSOS DO SISTEMA
Filesystem      Size  Used Avail Use% Mounted on
               total        used        free      shared  buff/cache   available
Mem:            12Gi       712Mi       9.1Gi       1.0Mi       2.9Gi        11Gi
Swap:             0B          0B          0B


## 2. Instalar Blender (Última Versão)

In [ ]:
import os
import urllib.request
import tarfile
from pathlib import Path

# Versão do Blender (ajuste para a versão desejada)
BLENDER_VERSION = "4.2.3"  # Última versão LTS
BLENDER_VERSION_SHORT = "4.2"

# URL de download
BLENDER_URL = f"https://download.blender.org/release/Blender{BLENDER_VERSION_SHORT}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_TAR = "/content/blender.tar.xz"
BLENDER_DIR = "/content/blender"

print(f"📥 Baixando Blender {BLENDER_VERSION}...")
print(f"URL: {BLENDER_URL}\n")

# Download com barra de progresso
def download_progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    percent = min(downloaded * 100 / total_size, 100)
    print(f"\rProgresso: {percent:.1f}% ({downloaded/(1024*1024):.1f}/{total_size/(1024*1024):.1f} MB)", end='')

urllib.request.urlretrieve(BLENDER_URL, BLENDER_TAR, download_progress)
print("\n✅ Download concluído!\n")

# Extrair
print("📦 Extraindo Blender...")
with tarfile.open(BLENDER_TAR, 'r:xz') as tar:
    tar.extractall('/content/')

# Renomear pasta
extracted_dir = f"/content/blender-{BLENDER_VERSION}-linux-x64"
if os.path.exists(extracted_dir):
    os.rename(extracted_dir, BLENDER_DIR)

# Limpar arquivo tar
os.remove(BLENDER_TAR)

# Definir executável
BLENDER_EXECUTABLE = f"{BLENDER_DIR}/blender"

# Verificar instalação
!{BLENDER_EXECUTABLE} --version

print(f"\n✅ Blender instalado com sucesso!")
print(f"📂 Localização: {BLENDER_DIR}")
print(f"🔧 Executável: {BLENDER_EXECUTABLE}")

📥 Baixando Blender 4.2.3...
URL: https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz

Progresso: 100.0% (335.6/335.6 MB)
✅ Download concluído!

📦 Extraindo Blender...


/tmp/ipython-input-1482519030.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/')


Blender 4.2.3 LTS
	build date: 2024-10-14
	build time: 23:31:34
	build commit date: 2024-10-14
	build commit time: 15:20
	build hash: 0e22e4fcea03
	build branch: blender-v4.2-release
	build platform: Linux
	build type: Release
	build c flags:  -Wall -Werror=implicit-function-declaration -Wstrict-prototypes -Werror=return-type -Werror=vla -Wmissing-prototypes -Wno-char-subscripts -Wno-unknown-pragmas -Wpointer-arith -Wunused-parameter -Wwrite-strings -Wlogical-op -Wundef -Winit-self -Wmissing-include-dirs -Wno-div-by-zero -Wtype-limits -Wformat-signedness -Wrestrict -Wno-stringop-overread -Wno-stringop-overflow -Wnonnull -Wabsolute-value -Wuninitialized -Wredundant-decls -Wshadow -Wimplicit-fallthrough=5 -Wno-error=unused-but-set-variable  -march=x86-64-v2 -std=gnu11 -pipe -fPIC -funsigned-char -fno-strict-aliasing -ffp-contract=off  
	build c++ flags:  -Wuninitialized -Wredundant-decls -Wall -Wno-invalid-offsetof -Wno-sign-compare -Wlogical-op -Winit-self -Wmissing-include-dirs -Wno-di

## 3. Instalar Dependências Python

In [ ]:
# Bibliotecas para processamento de imagens e vídeos
!pip install -q Pillow opencv-python-headless matplotlib tqdm

print("✅ Dependências instaladas!")

✅ Dependências instaladas!


## 4. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Definir caminhos
DRIVE_PATH = '/content/drive/MyDrive'
BLENDER_PROJECTS = f'{DRIVE_PATH}/blender-render'
RENDER_OUTPUT = f'{DRIVE_PATH}/blender-render'

# Criar pastas se não existirem
!mkdir -p {BLENDER_PROJECTS}
!mkdir -p {RENDER_OUTPUT}

print("✅ Google Drive montado!")
print(f"📁 Projetos: {BLENDER_PROJECTS}")
print(f"🎬 Renders: {RENDER_OUTPUT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive montado!
📁 Projetos: /content/drive/MyDrive/blender-render
🎬 Renders: /content/drive/MyDrive/blender-render


## 5. Script Python para Configuração de Renderização Fotorealista

In [ ]:
# Criar script de configuração Cycles
render_config_script = """
import bpy
import sys

def setup_cycles_photorealistic(samples=128, resolution_x=1080, resolution_y=1920, use_denoising=True):
    '''
    Configura o Cycles para renderização fotorealista com GPU
    '''
    scene = bpy.context.scene

    # Engine de renderização
    scene.render.engine = 'CYCLES'

    # Configurar Cycles
    cycles = scene.cycles

    # GPU acceleration
    cycles.device = 'GPU'

    # Samples (qualidade)
    cycles.samples = samples
    cycles.preview_samples = max(32, samples // 4)

    # Light paths (fotorealismo)
    cycles.max_bounces = 12
    cycles.diffuse_bounces = 4
    cycles.glossy_bounces = 4
    cycles.transmission_bounces = 12
    cycles.volume_bounces = 2
    cycles.transparent_max_bounces = 8

    # Caustics (efeitos de luz através de vidro/água)
    cycles.caustics_reflective = True
    cycles.caustics_refractive = True

    # Denoising (reduz ruído)
    if use_denoising:
        scene.cycles.use_denoising = True
        scene.cycles.denoiser = 'OPENIMAGEDENOISE'  # AI denoiser

    # Resolução
    scene.render.resolution_x = resolution_x
    scene.render.resolution_y = resolution_y
    scene.render.resolution_percentage = 100

    # Formato de saída
    scene.render.image_settings.file_format = 'PNG'
    scene.render.image_settings.color_mode = 'RGBA'
    scene.render.image_settings.color_depth = '16'  # 16-bit para maior qualidade
    scene.render.image_settings.compression = 15  # Compressão PNG

    # Film (transparência e exposure)
    scene.render.film_transparent = False
    scene.view_settings.view_transform = 'Filmic'
    scene.view_settings.look = 'Medium High Contrast'

    print(f"✅ Cycles configurado para renderização fotorealista")
    print(f"   Samples: {samples}")
    print(f"   Resolução: {resolution_x}x{resolution_y}")
    print(f"   Device: GPU")
    print(f"   Denoising: {use_denoising}")

# Configurar GPU preferences
def setup_gpu():
    preferences = bpy.context.preferences
    cycles_preferences = preferences.addons['cycles'].preferences

    # Ativar GPU
    cycles_preferences.compute_device_type = 'OPTIX'  # Melhor para NVIDIA

    # Detectar GPUs
    cycles_preferences.get_devices()

    # Ativar todas as GPUs disponíveis
    for device in cycles_preferences.devices:
        device.use = True
        print(f"   GPU detectada: {device.name} ({device.type})")

    print("✅ GPU configurada para Cycles")

if __name__ == "__main__":
    setup_gpu()

    # Parâmetros padrão (podem ser sobrescritos)
    samples = 128
    res_x = 1920
    res_y = 1080

    setup_cycles_photorealistic(samples, res_x, res_y)
"""

# Salvar script
RENDER_CONFIG_SCRIPT = "/content/setup_cycles.py"
with open(RENDER_CONFIG_SCRIPT, 'w') as f:
    f.write(render_config_script)

print("✅ Script de configuração criado!")
print(f"📄 {RENDER_CONFIG_SCRIPT}")

✅ Script de configuração criado!
📄 /content/setup_cycles.py


## 6. Função de Renderização Python

In [ ]:
import subprocess
import os
from pathlib import Path
import json

def render_blender(
    blend_file,
    output_path,
    samples=128,
    resolution=(1920, 1080),
    frame_start=None,
    frame_end=None,
    animation=False,
    engine='CYCLES',
    use_gpu=True,
    id=1
):
    """
    Renderiza arquivo .blend com configurações fotorealistas

    Args:
        blend_file: Caminho para arquivo .blend
        output_path: Caminho de saída para render
        samples: Número de samples (128-512 para fotorealismo, mais = melhor qualidade)
        resolution: Tupla (width, height)
        frame_start: Frame inicial (para animação)
        frame_end: Frame final (para animação)
        animation: True para animação, False para imagem única
        engine: 'CYCLES' (fotorealista) ou 'EEVEE' (rápido)
        use_gpu: Usar GPU para renderização
    """

    if not os.path.exists(blend_file):
        print(f"❌ Arquivo não encontrado: {blend_file}")
        return False

    # Criar diretório de saída
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Script Python para configurar renderização
    python_script = f"""
import bpy
scene = bpy.context.scene

# Engine
scene.render.engine = '{engine}'

# Cycles settings
if '{engine}' == 'CYCLES':
    scene.cycles.device = '{'GPU' if use_gpu else 'CPU'}'
    scene.cycles.samples = {samples}
    scene.cycles.use_denoising = True
    scene.cycles.denoiser = 'OPENIMAGEDENOISE'

    # Photorealistic settings
    scene.cycles.max_bounces = 12
    scene.cycles.diffuse_bounces = 4
    scene.cycles.glossy_bounces = 4
    scene.cycles.transmission_bounces = 12
    scene.cycles.caustics_reflective = True
    scene.cycles.caustics_refractive = True

    # GPU setup
    prefs = bpy.context.preferences.addons['cycles'].preferences
    prefs.compute_device_type = 'OPTIX'
    prefs.get_devices()
    for device in prefs.devices:
        device.use = True

# Resolution
scene.render.resolution_x = {resolution[0]}
scene.render.resolution_y = {resolution[1]}
scene.render.resolution_percentage = 100

# Output format
scene.render.image_settings.file_format = 'PNG'
scene.render.image_settings.color_mode = 'RGBA'
scene.render.image_settings.color_depth = '16'

# Animation frames
{'scene.frame_start = ' + str(frame_start) if frame_start else ''}
{'scene.frame_end = ' + str(frame_end) if frame_end else ''}

# Output path
scene.render.filepath = '{output_path}'

print(f"Configuração aplicada: {engine}, {samples} samples, {resolution[0]}x{resolution[1]}")
"""

    # Salvar script temporário
    script_file = f'/content/render_script{id}.py'
    with open(script_file, 'w') as f:
        f.write(python_script)

    # Comando Blender
    cmd = [
        BLENDER_EXECUTABLE,
        '--background',
        blend_file,
        '--python', script_file
    ]

    if animation:
        cmd.extend(['--render-anim'])
    else:
        cmd.extend(['--render-frame', '1'])

    print("="*60)
    print("🎬 INICIANDO RENDERIZAÇÃO")
    print("="*60)
    print(f"📁 Arquivo: {blend_file}")
    print(f"🎨 Engine: {engine}")
    print(f"🔢 Samples: {samples}")
    print(f"📐 Resolução: {resolution[0]}x{resolution[1]}")
    print(f"💾 Saída: {output_path}")
    print(f"🎞️  Tipo: {'Animação' if animation else 'Imagem única'}")
    if animation and frame_start and frame_end:
        print(f"📽️  Frames: {frame_start} - {frame_end}")
    print("="*60 + "\n")

    # Executar renderização
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True
        )

        # Mostrar output em tempo real
        for line in process.stdout:
            print(line, end='')

        process.wait()

        if process.returncode == 0:
            print("\n" + "="*60)
            print("✅ RENDERIZAÇÃO CONCLUÍDA COM SUCESSO!")
            print("="*60)
            return True
        else:
            print(f"\n❌ Erro na renderização (código {process.returncode})")
            return False

    except Exception as e:
        print(f"\n❌ Erro: {str(e)}")
        return False
    finally:
        # Limpar script temporário
        if os.path.exists(script_file):
            os.remove(script_file)

print("✅ Função de renderização configurada!")

✅ Função de renderização configurada!


## 7. Exemplo: Renderizar Projeto do Google Drive

In [ ]:
# Configurações de renderização
BLEND_FILE = f'{BLENDER_PROJECTS}/cyberpunk001.blend'  # ⚠️ AJUSTE O NOME DO ARQUIVO
OUTPUT_FILE = f'{RENDER_OUTPUT}/cyberpunk001/render_fotorealista.png'

# Parâmetros de qualidade
SAMPLES = 128  # 128=rápido, 256=bom, 512=excelente, 1024+=produção
RESOLUTION = (1080, 1920)  # FullHD, ou (3840, 2160) para 4K

# Verificar se arquivo existe
if os.path.exists(BLEND_FILE):
    print("🎬 Iniciando renderização fotorealista...\n")

    success = render_blender(
        blend_file=BLEND_FILE,
        output_path=OUTPUT_FILE,
        samples=SAMPLES,
        resolution=RESOLUTION,
        animation=False,  # True para animação
        engine='CYCLES',
        use_gpu=True
    )
else:
    print(f"⚠️ Arquivo não encontrado: {BLEND_FILE}")
    print(f"\n📋 Arquivos .blend disponíveis em {BLENDER_PROJECTS}:")
    !ls -lh {BLENDER_PROJECTS}/*.blend 2>/dev/null || echo "Nenhum arquivo .blend encontrado"

🎬 Iniciando renderização fotorealista...

🎬 INICIANDO RENDERIZAÇÃO
📁 Arquivo: /content/drive/MyDrive/blender-render/cyberpunk001.blend
🎨 Engine: CYCLES
🔢 Samples: 128
📐 Resolução: 1080x1920
💾 Saída: /content/drive/MyDrive/blender-render/cyberpunk001/render_fotorealista.png
🎞️  Tipo: Imagem única



KeyboardInterrupt: 

## 8. Exemplo: Renderizar Animação

In [ ]:
# Renderizar animação completa
BLEND_FILE = f'{BLENDER_PROJECTS}/cyberpunk001.blend'  # ⚠️ AJUSTE O NOME
ANIMATION_OUTPUT = f'{RENDER_OUTPUT}/cyberpunk001/frame_'  # Frames serão salvos como frame_0001.png, frame_0002.png, etc

# Criar pasta para frames
!mkdir -p {os.path.dirname(ANIMATION_OUTPUT)}

if os.path.exists(BLEND_FILE):
    success = render_blender(
        blend_file=BLEND_FILE,
        output_path=ANIMATION_OUTPUT,
        samples=128,  # Menos samples para animação (mais rápido)
        resolution=(1080, 1920),
        frame_start=76,  # Frame inicial
        frame_end=350,  # Frame final
        animation=True,
        engine='CYCLES',
        use_gpu=True
    )

    if success:
        print(f"\n🎞️  Frames salvos em: {os.path.dirname(ANIMATION_OUTPUT)}")
        !ls -lh {os.path.dirname(ANIMATION_OUTPUT)}
else:
    print(f"⚠️ Arquivo não encontrado: {BLEND_FILE}")

A saída de streaming foi truncada nas últimas 5000 linhas.
Fra:76 Mem:815.60M (Peak 941.43M) | Time:00:04.41 | Mem:489.82M, Peak:489.82M | Scene, ViewLayer | Updating Camera
Fra:76 Mem:815.60M (Peak 941.43M) | Time:00:04.41 | Mem:489.82M, Peak:489.82M | Scene, ViewLayer | Updating Meshes Flags
Fra:76 Mem:815.60M (Peak 941.43M) | Time:00:04.41 | Mem:489.82M, Peak:489.82M | Scene, ViewLayer | Updating Objects
Fra:76 Mem:815.60M (Peak 941.43M) | Time:00:04.41 | Mem:489.82M, Peak:489.82M | Scene, ViewLayer | Updating Objects | Copying Transformations to device
Fra:76 Mem:815.65M (Peak 941.43M) | Time:00:04.41 | Mem:489.86M, Peak:489.86M | Scene, ViewLayer | Updating Objects | Applying Static Transformations
Fra:76 Mem:815.65M (Peak 941.43M) | Time:00:04.41 | Mem:489.86M, Peak:489.86M | Scene, ViewLayer | Updating Particle Systems
Fra:76 Mem:815.65M (Peak 941.43M) | Time:00:04.41 | Mem:489.86M, Peak:489.86M | Scene, ViewLayer | Updating Particle Systems | Copying Particles to device
Fra:76 

KeyboardInterrupt: 

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BLEND_FILE = f'{BLENDER_PROJECTS}/cyberpunk001.blend'
FOLDER = f'{RENDER_OUTPUT}/cyberpunk001/'
ANIMATION_OUTPUT = f'{RENDER_OUTPUT}/cyberpunk001/frame_'
SAMPLES = 128
RESOLUTION = (1080, 1920)
ANIMATION = True
ENGINE = 'CYCLES'
USE_GPU = True

# Criar tarefas com intervalos de frames
# Configurações
TOTAL_FRAMES = 200
MAX_WORKERS = 3  # número de threads
FRAME_START = 1
FRAME_END = TOTAL_FRAMES

# Lista frames já renderizados
existing_frames = set()
for f in os.listdir(FOLDER):
    if f.startswith("frame_") and f.endswith(".png"):  # ou .exr/.jpg
        try:
            num = int(f.split("_")[1].split(".")[0])
            existing_frames.add(num)
        except ValueError:
            pass

# Gera lista de frames faltantes
all_frames = set(range(171, TOTAL_FRAMES + 1))
frames_to_render = sorted(list(all_frames - existing_frames))

# Cria tasks **uma por frame**
tasks = []
task_id = 0
for frame in frames_to_render:
    tasks.append(dict(
        blend_file=BLEND_FILE,
        output_path=FOLDER,
        samples=SAMPLES,
        resolution=RESOLUTION,
        frame_start=frame,
        frame_end=frame,  # apenas 1 frame por task
        animation=True,
        engine='CYCLES',
        use_gpu=True,
        id=task_id
    ))
    task_id += 1

print(f"Total de tasks: {len(tasks)}")

# Executar em paralelo
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(render_blender, **t) for t in tasks]
    for future in as_completed(futures):
        try:
            success = future.result()
            print("Status:", success)
        except Exception as e:
            print("Erro na task:", e)

Fra:193 Mem:1050.80M (Peak 1274.13M) | Time:01:24.87 | Remaining:02:20.76 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 44/128
Fra:193 Mem:1050.80M (Peak 1274.13M) | Time:01:30.63 | Remaining:01:30.09 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 60/128
Fra:194 Mem:1050.80M (Peak 1050.80M) | Time:01:07.60 | Remaining:02:04:25.39 | Mem:564.69M, Peak:564.69M | Scene, ViewLayer | Sample 1/128
Fra:194 Mem:1050.80M (Peak 1272.88M) | Time:01:09.44 | Remaining:01:03:39.69 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 2/128
Fra:194 Mem:1050.80M (Peak 1272.88M) | Time:01:11.07 | Remaining:43:13.83 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 3/128
Fra:194 Mem:1050.80M (Peak 1272.88M) | Time:01:13.05 | Remaining:33:11.16 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 4/128
Fra:193 Mem:1050.80M (Peak 1274.13M) | Time:01:36.96 | Remaining:00:58.72 | Mem:564.69M, Peak:714.99M | Scene, ViewLayer | Sample 76/128
Fra:194 Mem:1050.80M (Peak 1272.88M) | 

## 9. Converter Frames para Vídeo (FFmpeg) com Paralelização

In [ ]:
# Instalar FFmpeg
!apt-get install -y ffmpeg

def frames_to_video(frames_dir, output_video, fps=24, quality='high', start_frame=1):
    """
    Converte frames PNG/JPG em vídeo MP4

    Args:
        frames_dir: Diretório com os frames
        output_video: Caminho do vídeo de saída
        fps: Frames por segundo
        quality: 'high', 'medium', ou 'low'
        start_frame: Número do primeiro frame
    """

    crf_values = {'high': 18, 'medium': 23, 'low': 28}
    crf = crf_values.get(quality, 18)

    # Padrão de arquivos
    frame_pattern = f'{frames_dir}/%04d.png'

    # Comando FFmpeg com multi-threading
    cmd = [
        'ffmpeg', '-y',
        '-framerate', str(fps),
        '-start_number', str(start_frame),
        '-i', frame_pattern,
        '-c:v', 'libx264',
        '-preset', 'slow',
        '-crf', str(crf),
        '-pix_fmt', 'yuv420p',
        '-threads', '0',  # Usar todos os threads disponíveis
        '-movflags', '+faststart',  # Otimização para streaming
        output_video
    ]

    print("=" * 60)
    print("🎬 CONVERTENDO FRAMES PARA VÍDEO")
    print("=" * 60)
    print(f"📁 Frames: {frames_dir}")
    print(f"🎞️  FPS: {fps}")
    print(f"⚙️  Qualidade: {quality} (CRF {crf})")
    print(f"💾 Output: {output_video}")
    print("=" * 60 + "\n")

    # Verificar se há frames
    import glob
    frames = glob.glob(f'{frames_dir}/*.png')
    if not frames:
        print(f"❌ Nenhum frame encontrado em {frames_dir}")
        return False

    print(f"📊 Total de frames encontrados: {len(frames)}")
    print("🎬 Iniciando conversão...\n")

    start_time = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start_time

    if result.returncode == 0:
        print(f"\n✅ Vídeo criado em {elapsed:.1f}s: {output_video}")

        # Informações do vídeo
        print("\n📊 Informações do vídeo:")
        !ffprobe -v error -show_entries format=duration,size,bit_rate -show_entries stream=width,height,codec_name -of default=noprint_wrappers=1 {output_video}

        # Tamanho do arquivo
        file_size = os.path.getsize(output_video) / (1024 * 1024)
        print(f"\n💾 Tamanho: {file_size:.1f} MB")

        return True
    else:
        print(f"\n❌ Erro ao criar vídeo: {result.stderr}")
        return False

# Exemplo de uso com frames paralelos:
# frames_to_video(
#     frames_dir=ANIMATION_OUTPUT_DIR,
#     output_video=f'{RENDER_OUTPUT}/animacao_paralela.mp4',
#     fps=24,
#     quality='high',
#     start_frame=FRAME_START
# )

print("✅ Função de conversão com paralelização configurada!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
✅ Função de conversão com paralelização configurada!


In [ ]:
import time
import subprocess
import os

frames_to_video(
    frames_dir=f'{RENDER_OUTPUT}/cyberpunk001',
    output_video=f'{RENDER_OUTPUT}/cyberpunk001.mp4',
    fps=24,
    quality='high'
)

🎬 CONVERTENDO FRAMES PARA VÍDEO
📁 Frames: /content/drive/MyDrive/blender-render/cyberpunk001
🎞️  FPS: 24
⚙️  Qualidade: high (CRF 18)
💾 Output: /content/drive/MyDrive/blender-render/cyberpunk001.mp4

📊 Total de frames encontrados: 350
🎬 Iniciando conversão...


✅ Vídeo criado em 303.1s: /content/drive/MyDrive/blender-render/cyberpunk001.mp4

📊 Informações do vídeo:
codec_name=h264
width=1080
height=1920
duration=14.584000
size=29072273
bit_rate=15947489

💾 Tamanho: 27.7 MB


True

## 10. Criar Cena Demo Fotorealista

In [ ]:
# Script para criar cena demo com materiais fotorealistas
demo_script = """
import bpy
import math

# Limpar cena
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete()

# Adicionar plano (chão)
bpy.ops.mesh.primitive_plane_add(size=10, location=(0, 0, 0))
floor = bpy.context.active_object
floor.name = 'Floor'

# Material para o chão
floor_mat = bpy.data.materials.new(name='Floor_Material')
floor_mat.use_nodes = True
floor.data.materials.append(floor_mat)

# Configurar material do chão (mármore)
nodes = floor_mat.node_tree.nodes
nodes.clear()
bsdf = nodes.new('ShaderNodeBsdfPrincipled')
output = nodes.new('ShaderNodeOutputMaterial')
bsdf.inputs['Base Color'].default_value = (0.8, 0.8, 0.8, 1)
bsdf.inputs['Roughness'].default_value = 0.2
bsdf.inputs['Metallic'].default_value = 0.0
floor_mat.node_tree.links.new(bsdf.outputs['BSDF'], output.inputs['Surface'])

# Adicionar esferas com materiais diferentes
materials_data = [
    {'name': 'Metal', 'color': (0.8, 0.8, 0.8, 1), 'metallic': 1.0, 'roughness': 0.1, 'pos': (-3, 0, 1)},
    {'name': 'Glass', 'color': (1, 1, 1, 1), 'metallic': 0.0, 'roughness': 0.0, 'transmission': 1.0, 'ior': 1.45, 'pos': (0, 0, 1)},
    {'name': 'Plastic', 'color': (0.8, 0.2, 0.2, 1), 'metallic': 0.0, 'roughness': 0.3, 'pos': (3, 0, 1)},
]

for mat_data in materials_data:
    # Criar esfera
    bpy.ops.mesh.primitive_uv_sphere_add(radius=1, location=mat_data['pos'])
    sphere = bpy.context.active_object
    sphere.name = f'Sphere_{mat_data["name"]}'

    # Criar material
    mat = bpy.data.materials.new(name=mat_data['name'])
    mat.use_nodes = True
    sphere.data.materials.append(mat)

    # Configurar nodes
    nodes = mat.node_tree.nodes
    nodes.clear()
    bsdf = nodes.new('ShaderNodeBsdfPrincipled')
    output = nodes.new('ShaderNodeOutputMaterial')

    bsdf.inputs['Base Color'].default_value = mat_data['color']
    bsdf.inputs['Metallic'].default_value = mat_data['metallic']
    bsdf.inputs['Roughness'].default_value = mat_data['roughness']

    if 'transmission' in mat_data:
        bsdf.inputs['Transmission'].default_value = mat_data['transmission']
        bsdf.inputs['IOR'].default_value = mat_data['ior']

    mat.node_tree.links.new(bsdf.outputs['BSDF'], output.inputs['Surface'])

# Adicionar luz
bpy.ops.object.light_add(type='SUN', location=(5, 5, 10))
sun = bpy.context.active_object
sun.data.energy = 3
sun.rotation_euler = (math.radians(45), math.radians(45), 0)

# Adicionar câmera
bpy.ops.object.camera_add(location=(7, -7, 5))
camera = bpy.context.active_object
camera.rotation_euler = (math.radians(60), 0, math.radians(45))
bpy.context.scene.camera = camera

# Configurar world (HDRI básico)
world = bpy.context.scene.world
world.use_nodes = True
bg = world.node_tree.nodes['Background']
bg.inputs['Color'].default_value = (0.5, 0.7, 1.0, 1.0)
bg.inputs['Strength'].default_value = 0.5

print("✅ Cena demo criada com sucesso!")
print("   - Chão com material reflexivo")
print("   - 3 esferas: Metal, Vidro, Plástico")
print("   - Iluminação Sun")
print("   - Câmera posicionada")
"""

# Salvar script
demo_script_file = '/content/create_demo.py'
with open(demo_script_file, 'w') as f:
    f.write(demo_script)

# Criar arquivo .blend
demo_blend = f'{BLENDER_PROJECTS}/demo_photorealistic.blend'

print("🎨 Criando cena demo fotorealista...\n")

!{BLENDER_EXECUTABLE} --background --python {demo_script_file} --save {demo_blend}

print(f"\n✅ Cena demo criada: {demo_blend}")
print("\nVocê pode agora renderizá-la com a célula acima!")

## 11. Renderizar Cena Demo

In [ ]:
# Renderizar a cena demo
demo_blend = f'{BLENDER_PROJECTS}/demo_photorealistic.blend'
demo_output = f'{RENDER_OUTPUT}/demo_render.png'

if os.path.exists(demo_blend):
    print("🎬 Renderizando cena demo...\n")

    success = render_blender(
        blend_file=demo_blend,
        output_path=demo_output,
        samples=256,  # Alta qualidade para demo
        resolution=(1920, 1080),
        animation=False,
        engine='CYCLES',
        use_gpu=True
    )

    if success and os.path.exists(demo_output):
        print(f"\n📸 Demo renderizada: {demo_output}\n")

        # Visualizar
        from IPython.display import Image, display
        display(Image(filename=demo_output, width=900))
else:
    print("⚠️ Execute a célula anterior para criar a cena demo primeiro!")

## 12. Informações e Dicas de Otimização

In [ ]:
# Estatísticas e dicas
print("="*60)
print("📊 CONFIGURAÇÕES DE QUALIDADE RECOMENDADAS")
print("="*60)
print("\n🎨 SAMPLES (Cycles):")
print("   • 64-128:   Preview/Teste rápido")
print("   • 128-256:  Boa qualidade")
print("   • 256-512:  Alta qualidade")
print("   • 512-1024: Produção profissional")
print("   • 1024+:    Qualidade máxima (renders longos)")

print("\n📐 RESOLUÇÕES COMUNS:")
print("   • HD:       1280x720")
print("   • Full HD:  1920x1080")
print("   • 2K:       2560x1440")
print("   • 4K:       3840x2160")
print("   • 8K:       7680x4320 (⚠️ Muito pesado!)")

print("\n⚡ DICAS DE OTIMIZAÇÃO:")
print("   1. Use DENOISING para reduzir samples necessários")
print("   2. Ative OptiX para NVIDIA GPUs (mais rápido)")
print("   3. Light Clamping reduz fireflies")
print("   4. Adaptive Sampling economiza tempo")
print("   5. Para animações, use menos samples (128-256)")
print("   6. Renders estáticos podem usar 512+ samples")

print("\n🚀 RENDERIZAÇÃO PARALELA:")
print("   • GPU T4: Use 2 workers (máximo)")
print("   • CPU: Use max_workers=cpu_count()")
print("   • Speedup típico: 1.5-2x com GPU, 4-8x com CPU")
print("   • Ideal para animações com 50+ frames")
print("   • Cada worker usa ~2-4GB RAM")
print("   • Re-renderize frames falhados automaticamente")

print("\n🎯 FOTOREALISMO:")
print("   • Use HDRI lighting (IBL)")
print("   • Ative Caustics para vidro/água")
print("   • Use PBR materials (Principled BSDF)")
print("   • Max Bounces: 12+ para luz indireta")
print("   • Filmic color management")

print("\n⚠️ LIMITAÇÕES COLAB:")
print("   • Tempo máximo: 12 horas")
print("   • RAM: ~12GB (divida entre workers)")
print("   • VRAM (T4): 16GB")
print("   • Disk: ~100GB disponível")

print("\n💡 ESTRATÉGIAS PARA ANIMAÇÕES LONGAS:")
print("   1. Renderize em batches (ex: frames 1-100, 101-200)")
print("   2. Use checkpoints (salve progresso regularmente)")
print("   3. Reduza samples para preview (64-128)")
print("   4. Aumente para final render (256-512)")
print("   5. Use motion blur do Blender (reduz stuttering)")
print("\n" + "="*60)

# Calcular tempo estimado de render
def estimate_render_time(total_frames, samples, resolution, workers=2):
    """Estima tempo de renderização baseado em parâmetros"""

    # Tempo base por frame (em segundos) - valores aproximados para T4
    base_time = 30  # 30s para 128 samples em 1080p

    # Ajustes
    sample_factor = samples / 128
    res_factor = (resolution[0] * resolution[1]) / (1920 * 1080)
    parallel_factor = 1 / workers

    time_per_frame = base_time * sample_factor * res_factor * parallel_factor
    total_time = time_per_frame * total_frames

    print(f"\n⏱️  ESTIMATIVA DE TEMPO:")
    print(f"   Frames: {total_frames}")
    print(f"   Samples: {samples}")
    print(f"   Resolução: {resolution[0]}x{resolution[1]}")
    print(f"   Workers: {workers}")
    print(f"   ─────────────────────────")
    print(f"   Tempo por frame: ~{time_per_frame:.1f}s")
    print(f"   Tempo total: ~{total_time/60:.1f} min ({total_time/3600:.1f}h)")
    print(f"   ⚠️  Esta é uma estimativa aproximada!")

# Exemplo:
# estimate_render_time(total_frames=250, samples=128, resolution=(1920, 1080), workers=2)

# Verificar tempo restante de sessão
try:
    from google.colab import runtime
    runtime.unassign()
except:
    print("\n💡 Para ver tempo de sessão: Runtime → Manage sessions")

## 📚 Recursos Adicionais

### Documentação:
- [Blender Cycles Documentation](https://docs.blender.org/manual/en/latest/render/cycles/index.html)
- [Principled BSDF](https://docs.blender.org/manual/en/latest/render/shader_nodes/shader/principled.html)
- [GPU Rendering](https://docs.blender.org/manual/en/latest/render/cycles/gpu_rendering.html)

### HDRIs Gratuitos:
- [Poly Haven](https://polyhaven.com/hdris) - HDRIs gratuitos 8K
- [HDRI Haven](https://hdrihaven.com/) - Biblioteca gratuita

### Assets PBR Gratuitos:
- [Poly Haven](https://polyhaven.com/) - Texturas, modelos, HDRIs
- [ambientCG](https://ambientcg.com/) - Materiais PBR
- [Quixel Megascans](https://quixel.com/megascans) - Assets AAA (grátis com Unreal)

### Tutoriais:
- [Blender Guru - Donut Tutorial](https://www.youtube.com/playlist?list=PLjEaoINr3zgFX8ZsChQVQsuDSjEqdWMAD)
- [CG Geek](https://www.youtube.com/user/Blenderfan93)
- [Blender Official Tutorials](https://www.blender.org/support/tutorials/)

---

**🎉 Pronto para criar renders fotorealistas incríveis!**

Execute as células acima sequencialmente e ajuste os parâmetros conforme necessário para seu projeto.